# Memory-Split PoC — Optimal (GPT-5.6-sol) Retriever

**Question.** At a fixed parameter budget, does a model that *offloads facts* to an external retriever (**SPLIT**) beat a dense twin that stores them in-weights (**DENSE**) once retrieval is perfect — and does it reason better?

Facts are **real Wikidata triples** (PopQA); the retriever's answers are generated by **GPT-5.6-sol** (`openai-group/gpt-5.6-sol`) via the TrueFoundry gateway. The two arms share one corpus, model, budget, and init — the only difference is whether fact values carry training loss.

**Evals:**
- **Fact-QA** (single-hop): DENSE @ closed-book, DENSE + oracle (RAG), SPLIT @ GPT-oracle, + SPLIT @ gold upper bound. Graded by string-match **and** an LLM judge (`--judge`, credits paraphrases/aliases exact-match misses).
- **Reason-over-facts** (in-context): GPT-phrased compositional yes/no questions over real fact pairs; BOTH models get the facts in context and must combine them. Tests reasoning capacity given identical facts.
- **Knowledge-free reasoning** composite (iGSM + deduction), supporting.

**Scale.** Team's primary scale **d160m (~162M)** by default; `--model d360m` (~356M, needs A100).

**Runtime:** GPU (A100 recommended). Run top to bottom; paste a TrueFoundry token when prompted.

**Caveats:** held-out fact-QA is partly definitional (dense never saw those facts); single-hop fact-QA tests *fact access*, not reasoning; the reason-over-facts and composite numbers are indicative (far smaller token budget than the team's cluster runs), and at this scale may sit near the majority baseline.

## 1. Get the code (always syncs to the latest branch state)

In [ ]:
import os
REPO_URL = "https://github.com/sidvenkatayogi/Memory-Split.git"
BRANCH = "poc/optimal-retriever"
if os.path.basename(os.getcwd()) != "Memory-Split":
    if not os.path.isdir("Memory-Split"):
        !git clone --branch {BRANCH} --single-branch {REPO_URL}
    %cd Memory-Split
# Sync to the latest branch code so a re-run picks up new evals, not a stale
# cached clone. Uses FETCH_HEAD to avoid the slash in the branch name; data/ is
# gitignored so your local corpus/checkpoints are untouched.
!git fetch -q origin {BRANCH} && git reset --hard -q FETCH_HEAD
!git log --oneline -1

In [ ]:
# Colab preinstalls torch/numpy/matplotlib/pyyaml with a CUDA-matched build.
# Install ONLY the missing runtime deps so we never reinstall (and risk
# breaking GPU) torch. `datasets` is not needed at runtime (facts are committed).
!pip install -q tiktoken openai
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Persist to Google Drive (checkpoints + golden cache + results)
Points `POC_PERSIST_DIR` at Drive so checkpoints, the (paid) GPT caches, and results **survive a disconnect** — a re-run reuses trained models and cached GPT answers instead of redoing them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['POC_PERSIST_DIR'] = '/content/drive/MyDrive/memory_split_poc'
os.makedirs(os.environ['POC_PERSIST_DIR'], exist_ok=True)
print('persist dir ->', os.environ['POC_PERSIST_DIR'])

## 3. TrueFoundry credentials

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("TrueFoundry token: ")
os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
os.environ["POC_GPT_MODEL"] = "openai-group/gpt-5.6-sol"

import sys; sys.path.insert(0, ".")
from evals.gpt_oracle import GatewayClient
print("gateway smoke ->", GatewayClient().smoke())  # e.g. 'Paris'

## 4. Build the shared corpus (offline; fast; regenerates the local corpus + eval sets)

In [ ]:
!python scripts/poc_run.py --stage build

## 5. Train the matched twins (reuses existing checkpoints — no retrain)
If a finished checkpoint for this `--model`/`--steps` is on Drive, training is **skipped** and the trained model is reused. `--fresh` forces a retrain; raise `--steps` to train longer.

In [ ]:
!python scripts/poc_run.py --stage train --device auto --model d160m --steps 4000 --ckpt-minutes 5

## 6. Generate golden knowledge with GPT-5.6-sol (cached)

In [ ]:
!python scripts/poc_run.py --stage gen-golden

## 7. Evaluate + report
Fact-QA (4 conditions, string-match **and** `--judge` LLM-grading), reason-over-facts (compositional, both arms in-context), and the knowledge-free composite. All use the trained models on Drive.

In [ ]:
!python scripts/poc_run.py --stage eval   --device auto --gold-oracle --judge
!python scripts/poc_run.py --stage reason --device auto
!python scripts/poc_run.py --stage report

In [ ]:
import json, os
from IPython.display import Image, display
persist = os.environ.get('POC_PERSIST_DIR', 'data/poc')
print(json.dumps(json.load(open(f'{persist}/poc_results.json')), indent=2))
display(Image(f'{persist}/poc_figure.png'))